# Lemma on Colab: math RAG with LFM3 running locally on the GPU

This notebook runs the [Lemma](https://github.com/DS-PNQ/mandatoryslopragapp) RAG app end to end inside a Colab runtime:

1. **llama.cpp** is built with CUDA for whatever GPU the runtime has (T4 on the free tier, or L4/A100) and serves **LFM3 1.7B** (GGUF) as an OpenAI-compatible endpoint on `127.0.0.1:8080`.
2. The **Next.js app** is built and started on port 3000 with `LFM3_BASE_URL` pointing at that server, so every answer is generated on the Colab GPU. No API key is needed.
3. You open the app through **Colab's own port proxy**. That link only works for you, signed in to Google. A public **Cloudflare quick tunnel** link is also available, and you can ask questions from inside the notebook.

**Before you run it:** *Runtime → Change runtime type → T4 GPU* (or any GPU), then *Runtime → Run all*.

**Build time.** Compiling llama.cpp takes about 20–25 minutes on the free tier's 2 CPUs, and less on larger runtimes. With `CACHE_TO_DRIVE = True` the compiled binary is saved to your Google Drive, so later sessions on the same GPU type skip the build. Colab will ask you to allow Drive access once.

The app runs with **LFM3 as its only model**. Hosted model keys are stripped from its environment, so nothing leaves the runtime except the one-time downloads. Optional: an `HF_TOKEN` secret (the key icon in the left sidebar, with notebook access on) speeds up the Hugging Face download.

## 1. Settings

In [ ]:
# --- What to run -----------------------------------------------------------
REPO_URL    = "https://github.com/DS-PNQ/mandatoryslopragapp.git"
REPO_BRANCH = ""                     # "" = the repository's default branch

GGUF_REPO   = "LiquidAI/LFM3-1.7B-GGUF"
# First quantization found in the repo wins. A 1.7B model at Q8_0 is ~1.8 GB,
# which any Colab GPU holds easily.
GGUF_QUANTS = ["Q8_0", "F16", "BF16", "Q6_K", "Q5_K_M", "Q4_K_M"]

LLAMA_CPP_REF  = "latest"            # "latest" release tag, "master", or a tag like "b12723"
CACHE_TO_DRIVE = True                # keep the compiled llama-server in Google Drive between sessions
DRIVE_CACHE    = "/content/drive/MyDrive/lemma-cache"

# --- llama-server ----------------------------------------------------------
# The app asks for up to 12,000 new tokens per answer plus ~3k tokens of
# passages, so each slot needs ~16k context.
PARALLEL_SLOTS = 2
CTX_TOTAL      = 16384 * PARALLEL_SLOTS

# --- Ports / access ----------------------------------------------------------
LLAMA_PORT  = 8080
APP_PORT    = 3000
OPEN_TUNNEL = True                   # also print a public trycloudflare.com link

WORK = "/content/lemma"

## 2. Helpers and environment check

In [ ]:
import glob, json, os, re, shutil, subprocess, sys, time, urllib.request
from pathlib import Path

WORK = Path(WORK); WORK.mkdir(parents=True, exist_ok=True)
LOGS = WORK / "logs"; LOGS.mkdir(exist_ok=True)

def sh(cmd, cwd=None, env=None, quiet=False):
    """Run a shell command, stream its output, raise if it fails."""
    print(f"$ {cmd}", flush=True)
    p = subprocess.Popen(cmd, shell=True, cwd=cwd, env=env, text=True,
                         stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    tail = []
    for line in p.stdout:
        tail = (tail + [line])[-40:]
        if not quiet:
            print(line, end="", flush=True)
    if p.wait() != 0:
        if quiet:
            print("".join(tail))
        raise RuntimeError(f"command failed ({p.returncode}): {cmd}")

def secret(name):
    """Colab secret (if notebook access is on), then environment variable, else None."""
    try:
        from google.colab import userdata
        v = userdata.get(name)
        if v:
            return v
    except Exception:
        pass
    return os.environ.get(name) or None

def wait_http(url, timeout, proc=None, log=None, ok=lambda r: r.status == 200):
    t0 = time.time()
    while time.time() - t0 < timeout:
        if proc is not None and proc.poll() is not None:
            print(Path(log).read_text()[-4000:] if log else "")
            raise RuntimeError(f"process exited with code {proc.returncode} before {url} came up")
        try:
            with urllib.request.urlopen(url, timeout=5) as r:
                if ok(r):
                    return round(time.time() - t0)
        except Exception:
            pass
        time.sleep(2)
    raise TimeoutError(f"{url} not ready after {timeout}s")

smi = subprocess.run("nvidia-smi --query-gpu=name,compute_cap,memory.total,driver_version --format=csv,noheader",
                     shell=True, capture_output=True, text=True).stdout.strip().splitlines()
assert smi, "No GPU. Runtime -> Change runtime type -> pick a GPU, then run again."
GPU_NAME, COMPUTE_CAP, GPU_MEM, DRIVER = [x.strip() for x in smi[0].split(",")]
CUDA_ARCH = COMPUTE_CAP.replace(".", "")                 # "7.5" -> "75"
print(f"GPU: {GPU_NAME} (sm_{CUDA_ARCH}, {GPU_MEM}), driver {DRIVER}, {os.cpu_count()} CPUs")

NVCC = shutil.which("nvcc") or ("/usr/local/cuda/bin/nvcc" if Path("/usr/local/cuda/bin/nvcc").exists() else None)
CUDA_VERSION = (re.search(r"release ([\d.]+)", subprocess.run(f"{NVCC} --version", shell=True, capture_output=True,
                text=True).stdout) or [None, "unknown"])[1] if NVCC else "none"
print("nvcc:", NVCC, "| CUDA", CUDA_VERSION)

## 3. Node.js 22

Next.js 16 needs Node 20.9 or newer, and the Node that ships with Colab is older. This installs the current Node 22 LTS into `/content/lemma/node`.

In [ ]:
NODE_DIR = WORK / "node"
if not (NODE_DIR / "bin" / "node").exists():
    releases = json.load(urllib.request.urlopen("https://nodejs.org/dist/index.json"))
    v = next(r["version"] for r in releases if r["version"].startswith("v22."))
    tarball = f"node-{v}-linux-x64.tar.xz"
    sh(f"curl -fsSL https://nodejs.org/dist/{v}/{tarball} -o {WORK}/{tarball}")
    sh(f"mkdir -p {NODE_DIR} && tar -xJf {WORK}/{tarball} -C {NODE_DIR} --strip-components=1 && rm {WORK}/{tarball}")
os.environ["PATH"] = f"{NODE_DIR}/bin:" + os.environ["PATH"]
sh("node --version && npm --version")

## 4. llama.cpp with CUDA

Built as a single `llama-server` binary for this GPU's architecture only, which keeps compile time down. The Drive cache is keyed by llama.cpp version, GPU architecture and CUDA version, so a cached binary is only reused where it will run.

In [ ]:
ref = LLAMA_CPP_REF
if ref == "latest":
    # Release tags are bNNNNN; git ls-remote avoids the rate-limited GitHub API.
    out = subprocess.run("git ls-remote --tags --refs https://github.com/ggml-org/llama.cpp 'b*'",
                         shell=True, capture_output=True, text=True).stdout
    tags = re.findall(r"refs/tags/(b\d+)$", out, re.M)
    ref = max(tags, key=lambda t: int(t[1:])) if tags else "master"
print("llama.cpp", ref)

LLAMA_SERVER = str(WORK / "llama-server")
cache_name = f"llama-server-{ref}-sm{CUDA_ARCH}-cuda{CUDA_VERSION}"
cached = None
if CACHE_TO_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    Path(DRIVE_CACHE).mkdir(parents=True, exist_ok=True)
    cached = Path(DRIVE_CACHE) / cache_name

if cached is not None and cached.exists() and ref != "master":
    print("Using cached build from Drive:", cached)
    shutil.copy(cached, LLAMA_SERVER)                     # Drive does not keep the exec bit
    os.chmod(LLAMA_SERVER, 0o755)
elif not Path(LLAMA_SERVER).exists():
    assert NVCC, "CUDA toolkit (nvcc) not found; use a GPU runtime."
    src = WORK / "llama.cpp"
    if not src.exists():
        sh(f"git clone --depth 1 --branch {ref} https://github.com/ggml-org/llama.cpp {src}")
    build_env = {**os.environ, "PATH": f"{Path(NVCC).parent}:" + os.environ["PATH"], "CUDACXX": NVCC}

    # ggml links the CUDA driver library (CUDA::cuda_driver). CMake looks for an
    # unversioned libcuda.so, which some images lack, so point it at the toolkit
    # stub or the driver's libcuda.so.1 explicitly.
    cuda_home = Path(NVCC).resolve().parent.parent
    candidates = [*glob.glob(f"{cuda_home}/lib64/stubs/libcuda.so"),
                  *glob.glob(f"{cuda_home}/targets/*/lib/stubs/libcuda.so"),
                  *glob.glob("/usr/local/cuda*/lib64/stubs/libcuda.so"),
                  *glob.glob("/usr/local/cuda*/targets/*/lib/stubs/libcuda.so")]
    candidates += re.findall(r"=>\s*(\S*libcuda\.so(?:\.1)?)$",
                             subprocess.run("ldconfig -p", shell=True, capture_output=True, text=True).stdout, re.M)
    candidates += glob.glob("/usr/lib64-nvidia/libcuda.so*") + glob.glob("/usr/lib/x86_64-linux-gnu/libcuda.so*")
    libcuda = next((c for c in candidates if os.path.isfile(c)), None)
    print("libcuda:", libcuda or "not found, building without VMM")
    driver_flag = f"-DCUDA_cuda_driver_LIBRARY={libcuda}" if libcuda else "-DGGML_CUDA_NO_VMM=ON"

    shutil.rmtree(src / "build", ignore_errors=True)        # drop a cache left by a failed configure
    sh(f"cmake -B build -DGGML_CUDA=ON -DCMAKE_CUDA_ARCHITECTURES={CUDA_ARCH} -DBUILD_SHARED_LIBS=OFF "
       f"{driver_flag} -DLLAMA_CURL=OFF -DLLAMA_BUILD_TESTS=OFF -DCMAKE_BUILD_TYPE=Release",
       cwd=src, env=build_env, quiet=True)
    t0 = time.time()
    print(f"Compiling on {os.cpu_count()} CPUs; this is the slow step.", flush=True)
    sh(f"cmake --build build --config Release --target llama-server -j{os.cpu_count()}", cwd=src, env=build_env, quiet=True)
    print(f"built {ref} in {(time.time() - t0) / 60:.1f} min")
    shutil.copy(src / "build" / "bin" / "llama-server", LLAMA_SERVER)
    if cached is not None and ref != "master":
        shutil.copy(LLAMA_SERVER, cached)
        print("Saved to Drive:", cached)

sh(f"{LLAMA_SERVER} --version")

## 5. Download LFM3 1.7B (GGUF)

In [ ]:
from huggingface_hub import hf_hub_download, list_repo_files

HF_TOKEN = secret("HF_TOKEN")
files = [f for f in list_repo_files(GGUF_REPO, token=HF_TOKEN) if f.lower().endswith(".gguf")]
print("GGUF files in", GGUF_REPO, "->", files)

def pick(files, quants):
    for q in quants:
        for f in files:
            if re.search(rf"[-_.]{re.escape(q)}\.gguf$", f, re.I) and "mmproj" not in f.lower():
                return q, f
    raise FileNotFoundError(f"none of {quants} in {GGUF_REPO}")

QUANT, GGUF_FILE = pick(files, GGUF_QUANTS)
MODEL_PATH = hf_hub_download(GGUF_REPO, GGUF_FILE, token=HF_TOKEN, local_dir=str(WORK / "models"))
MODEL_NAME = f"LFM3-1.7B-{QUANT}"
print(f"{MODEL_NAME}: {MODEL_PATH} ({os.path.getsize(MODEL_PATH) / 2**30:.2f} GiB)")

## 6. Start llama-server

`--jinja` applies LFM3's own chat template, which opens a reasoning block. llama-server returns that trace in `reasoning_content`, and the app shows it as a progress line rather than as answer text.

In [ ]:
LLAMA_LOG = LOGS / "llama-server.log"
subprocess.run(["pkill", "-f", f"llama-server.*--port {LLAMA_PORT}"])

llama_cmd = [
    LLAMA_SERVER, "-m", MODEL_PATH, "--alias", MODEL_NAME, "--jinja",
    "--host", "127.0.0.1", "--port", str(LLAMA_PORT),
    "-ngl", "999", "-c", str(CTX_TOTAL), "-np", str(PARALLEL_SLOTS),
]
llama_proc = subprocess.Popen(llama_cmd, stdout=open(LLAMA_LOG, "w"), stderr=subprocess.STDOUT, start_new_session=True)
secs = wait_http(f"http://127.0.0.1:{LLAMA_PORT}/health", 600, llama_proc, LLAMA_LOG)
print(f"llama-server ready in {secs}s")
sh(f"grep -iE 'CUDA[0-9].*(model buffer|KV)|offloaded|n_ctx_slot|n_slots' {LLAMA_LOG} | head -20 || true")
sh("nvidia-smi --query-gpu=memory.used,memory.total --format=csv")

In [ ]:
# Smoke test straight against llama-server (OpenAI-compatible API).
req = urllib.request.Request(
    f"http://127.0.0.1:{LLAMA_PORT}/v1/chat/completions",
    data=json.dumps({"model": MODEL_NAME, "max_tokens": 512, "temperature": 1.0, "top_p": 0.95, "top_k": 20,
                     "messages": [{"role": "user", "content": "What is the derivative of x^3? One line."}]}).encode(),
    headers={"content-type": "application/json"})
t0 = time.time()
r = json.load(urllib.request.urlopen(req, timeout=300))
msg = r["choices"][0]["message"]
print("reasoning:", (msg.get("reasoning_content") or "")[:300].replace("\n", " "), "...")
print("answer   :", msg.get("content"))
print(f"{r['usage']['completion_tokens']} tokens in {time.time() - t0:.1f}s")

## 7. Build and start the app

In [ ]:
APP = WORK / "app"
if not (APP / "package.json").exists():
    branch = f"--branch {REPO_BRANCH}" if REPO_BRANCH else ""
    sh(f"git clone --depth 1 {branch} {REPO_URL} {APP}")
sh("git log --oneline -1", cwd=APP)

# LFM3 on this runtime is the only model: drop any hosted-model keys so the
# app never offers Opus or routes LFM3 through OpenRouter.
HOSTED_KEYS = {"ANTHROPIC_API_KEY", "ANTHROPIC_AUTH_TOKEN", "OPENROUTER_API_KEY", "LFM3_API_KEY"}
APP_ENV = {k: v for k, v in os.environ.items() if k not in HOSTED_KEYS}
APP_ENV.update({
    "NEXT_TELEMETRY_DISABLED": "1",
    "NODE_ENV": "production",
    "LFM3_BASE_URL": f"http://127.0.0.1:{LLAMA_PORT}/v1",
    "LFM3_MODEL": MODEL_NAME,
})

sh("npm ci --no-audit --no-fund", cwd=APP, env={**APP_ENV, "NODE_ENV": "development"}, quiet=True)
sh("npm run build", cwd=APP, env=APP_ENV, quiet=True)   # builds the e-means/BM25 index, then next build

In [ ]:
APP_LOG = LOGS / "app.log"
subprocess.run(["pkill", "-f", f"next start -p {APP_PORT}"])
app_proc = subprocess.Popen(f"npx next start -p {APP_PORT} -H 0.0.0.0", shell=True, cwd=APP, env=APP_ENV,
                            stdout=open(APP_LOG, "w"), stderr=subprocess.STDOUT, start_new_session=True)
secs = wait_http(f"http://127.0.0.1:{APP_PORT}/", 180, app_proc, APP_LOG)
print(f"Lemma is up on port {APP_PORT} ({secs}s)")

## 8. Ask from the notebook

Calls the app's `/api/ask` endpoint (the same one the web page uses), prints its status lines as they stream, and renders the answer with its sources.

In [ ]:
from IPython.display import Markdown, display

def ask(question, engine="lfm3", history=None, show_sources=True):
    body = json.dumps({"question": question, "engine": engine, "history": history or []}).encode()
    req = urllib.request.Request(f"http://127.0.0.1:{APP_PORT}/api/ask", data=body,
                                 headers={"content-type": "application/json"})
    sources, trace, transcript, last_status, t0 = [], None, "", None, time.time()
    with urllib.request.urlopen(req, timeout=900) as r:
        for raw in r:
            if not raw.strip():
                continue
            ev = json.loads(raw)
            if ev["type"] == "a2ui":
                upd = ev["message"].get("updateDataModel")
                if upd and upd["path"] == "/status" and upd["value"]["text"] != last_status:
                    last_status = upd["value"]["text"]
                    print(f"[{time.time() - t0:5.1f}s] {last_status}", flush=True)
                elif upd and upd["path"] == "/sources":
                    sources = upd["value"]
                elif upd and upd["path"] == "/trace":
                    trace = upd["value"]
            elif ev["type"] == "error":
                raise RuntimeError(ev["message"])
            elif ev["type"] == "done":
                transcript = ev["transcript"]
    if trace:
        print(f"model {trace['model']} | scanned {trace['scanned']}/{trace['total']} passages "
              f"in {len(trace['cells'])}/{trace['k']} e-means cells"
              + (f" | dense recall {trace['denseRecall']:.2f}" if trace.get("denseRecall") is not None else ""))
    display(Markdown(transcript or "_(empty answer)_"))
    if show_sources and sources:
        display(Markdown("\n".join(f"- **[{s['n']}]** {s['note']} §{s['section']} {s['heading']}" for s in sources)))
    return {"question": question, "answer": transcript}

turn = ask("Why does the geometric series converge exactly when |r| < 1?")

In [ ]:
# A follow-up that uses the previous turn as context.
ask("How does the ratio test generalise that?", history=[turn])

## 9. Open the web UI

**Colab proxy (private):** the first link goes through Colab's port proxy, so it only works in a browser signed in to the same Google account. If it shows a blank page or a cookie error, allow third-party cookies for `colab.research.google.com` and `*.googleusercontent.com`, or use the tunnel link.

**Cloudflare quick tunnel (public):** the second link works from any browser, but anyone who has it can use the app while the runtime is up. Stop it with the last cell.

In [ ]:
from google.colab.output import eval_js
PROXY_URL = eval_js(f"google.colab.kernel.proxyPort({APP_PORT}, {{'cache': false}})")
links = [f"**Colab proxy (only you):** [{PROXY_URL}]({PROXY_URL})"]

if OPEN_TUNNEL:
    CF = WORK / "cloudflared"
    if not CF.exists():
        sh(f"curl -fsSL https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -o {CF} && chmod +x {CF}")
    TUNNEL_LOG = LOGS / "cloudflared.log"
    subprocess.run(["pkill", "-f", "cloudflared tunnel"])
    tunnel_proc = subprocess.Popen([str(CF), "tunnel", "--no-autoupdate", "--url", f"http://127.0.0.1:{APP_PORT}"],
                                   stdout=open(TUNNEL_LOG, "w"), stderr=subprocess.STDOUT, start_new_session=True)
    PUBLIC_URL = None
    for _ in range(60):
        m = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", TUNNEL_LOG.read_text())
        if m:
            PUBLIC_URL = m.group(0); break
        time.sleep(1)
    if PUBLIC_URL:
        try:
            wait_http(PUBLIC_URL, 60)      # the hostname takes a few seconds to resolve
        except Exception:
            pass
        links.append(f"**Cloudflare tunnel (anyone with the link):** [{PUBLIC_URL}]({PUBLIC_URL})")
    else:
        print("Tunnel did not start:\n", TUNNEL_LOG.read_text()[-2000:])

display(Markdown("### Lemma is live\n\n" + "\n\n".join(links) +
                 "\n\nEvery answer is generated by LFM3 1.7B on this runtime's GPU."))

## 10. Logs and shutdown

In [ ]:
# Tail the logs if something looks off.
for log in [LOGS / "llama-server.log", LOGS / "app.log", LOGS / "cloudflared.log"]:
    if log.exists():
        print(f"===== {log.name} =====")
        print("".join(log.read_text().splitlines(keepends=True)[-25:]))

In [ ]:
# Stop everything (Runtime -> Disconnect and delete runtime also works).
for pattern in ["cloudflared tunnel", f"next start -p {APP_PORT}", f"llama-server.*--port {LLAMA_PORT}"]:
    subprocess.run(["pkill", "-f", pattern])
print("stopped")